In [ ]:
import abc
from typing import List, Dict, Tuple

class BaseEvalTask(abc.ABC):
    """Lớp nền tảng cho mọi tác vụ đánh giá (Benchmark Task)."""
    
    def __init__(self, dataset_name: str, split: str = "test", num_shots: int = 0):
        self.dataset_name = dataset_name
        self.split = split
        self.num_shots = num_shots
        # Danh sách chứa các ví dụ mẫu mực. Các class con sẽ điền giá trị vào đây.
        self.golden_shots = [] 

    @abc.abstractmethod
    def load_data(self) -> Tuple[List[Dict], List[str]]:
        """
        Tải dữ liệu.
        Returns: Tuple(List[examples], List[gold_answers])
        """
        pass

    @abc.abstractmethod
    def build_prompt(self, problem: str, few_shot_examples: List[Dict]) -> str:
        """Tạo prompt tùy chỉnh theo từng Task."""
        pass

    @abc.abstractmethod
    def evaluate_correctness(self, prediction: str, gold_answer: str) -> bool:
        """Chấm điểm đúng/sai theo chuẩn của bộ dữ liệu."""
        pass
        
    def generate_few_shots(self) -> List[Dict]:
        """
        Lấy các ví dụ Golden Shots đã được hard-code sẵn trong class con.
        Đảm bảo không lấy vượt quá self.num_shots.
        """
        if not self.golden_shots or self.num_shots == 0:
            return []
            
        return self.golden_shots[:self.num_shots]

In [ ]:
import re
from datasets import load_dataset
from typing import List, Dict, Tuple
from evaluation.tasks.base_task import BaseEvalTask
from evaluation.metrics import check_correctness

class CMATHTask(BaseEvalTask):
    def __init__(self, num_shots: int = 6):
        super().__init__(dataset_name="weitianwen/cmath", split="test", num_shots=num_shots)
        
        self.system_prompt = "你是一个优秀的数学助手。请一步步思考并解决以下数学问题，最后将最终答案放在 \\boxed{} 中。"
        
        self.golden_shots = [
            {
                "question": "芳芳买了一本书有99页，看了90页，她还剩多少页没有看？", 
                "answer": "还剩的没有看的页数=书的总页数-芳芳看了的页数，99-90=9。所以答案是：9。 \\boxed{9}"
            },
            {
                "question": "张师傅上午修了18把椅子，下午修了29把椅子，一天共修了多少把椅子？", 
                "answer": "一天共修的椅子数量=上午修的椅子数量+下午修的椅子数量，18+29=47。所以答案是：47。 \\boxed{47}"
            },
            {
                "question": "小猴摘了84个桃子，平均分给6只猴子，每只猴子能吃到几个桃子？", 
                "answer": "每只猴子能吃到的桃子数=总桃子数/猴子的数量，84/6=14。所以答案是：14。 \\boxed{14}"
            },
            {
                "question": "用面包机烤面包时，第一面烤2分钟，第二面只要烤1分钟，即烤一片面包需要3分钟，小勤的面包机一次只能放2片，他每天早上吃3片面包，至少需要烤多少分钟？", 
                "answer": "可以现将两片面包放入面包机烤2分钟，再将其中一片拿出来，将第三片面包放进去，烤1分钟，这样第一片面包就烤好了，将第一片面包拿出来将第二片面包放进去，继续烤1分钟，于是第二片面包也烤好了将其拿出来，第三片面包再烤1分钟也就烤好了，一共是2+1+1=5。所以答案是：5。 \\boxed{5}"
            },
            {
                "question": "一组学生植树，每人栽6棵还剩4棵；如果其中3人各栽5棵，其余每人各栽7棵，正好栽完。这一组学生有多少人？", 
                "answer": "假设学生的数量是x，每人栽6棵还剩4棵，也就是说树苗的数量=6x+4，又知道如果其中3人各栽5棵，其余每人各栽7棵，正好栽完，即6x+4=3*5+(x-3)*7，化简方程得到：x=10。所以答案是：10。 \\boxed{10}"
            },
            {
                "question": "某小学在“献爱心--为汶川地震区捐款”活动中，六年级五个班共捐款8000元，其中一班捐款1500元，二班比一班多捐款200元，三班捐款1600元，四班与五班捐款数之比是3：5。四班捐款多少元？", 
                "answer": "一班捐款1500元，而二班比一班多捐200元，所以二班捐款1500+200=1700元，又知道六年级五个班一共捐款8000元，所以四班和五班捐款之和 = 一共捐款 - 一班和二班和三班捐款之和，即8000-1500-1700-1600=3200元，而题目说四班与五班捐款数之比是3：5，则四班捐款了3200/(3+5)*3=1200元。所以答案是：1200。 \\boxed{1200}"
            }
        ]

    def load_data(self) -> Tuple[List[Dict], List[str]]:
        ds = load_dataset(self.dataset_name, split=self.split)
        return [{"question": item["question"]} for item in ds], [item["golden"] for item in ds]

    def build_prompt(self, problem: str, few_shot_examples: List[Dict]) -> str:
        prompt = f"<|im_start|>system\n{self.system_prompt}<|im_end|>\n"
        for ex in few_shot_examples:
            prompt += f"<|im_start|>user\n{ex['question']}<|im_end|>\n<|im_start|>assistant\n{ex['answer']}<|im_end|>\n"
        prompt += f"<|im_start|>user\n{problem}<|im_end|>\n<|im_start|>assistant\n"
        return prompt

    def evaluate_correctness(self, prediction: str, gold_answer: str) -> bool:

        is_correct = check_correctness(prediction, gold_answer)
        if is_correct:
            return True
            
        # fallback
        fallback_pattern = r"所以答案是[:：]\s*([0-9\.\-\/]+)"
        matches = re.findall(fallback_pattern, prediction)
        
        if matches:
            pred_ans = matches[-1].strip()
            return pred_ans.replace(",", "") == gold_answer.replace(",", "")
            
        return False

<|im_start|>system
你是一个优秀的数学助手。请一步步思考并解决以下数学问题，最后将最终答案放在 \boxed{} 中。<|im_end|>
<|im_start|>user
芳芳买了一本书有99页，看了90页，她还剩多少页没有看？<|im_end|>
<|im_start|>assistant
还剩的没有看的页数=书的总页数-芳芳看了的页数，99-90=9。所以答案是：9。 \boxed{9}<|im_end|>
<|im_start|>user
张师傅上午修了18把椅子，下午修了29把椅子，一天共修了多少把椅子？<|im_end|>
<|im_start|>assistant
一天共修的椅子数量=上午修的椅子数量+下午修的椅子数量，18+29=47。所以答案是：47。 \boxed{47}<|im_end|>
<|im_start|>user
小猴摘了84个桃子，平均分给6只猴子，每只猴子能吃到几个桃子？<|im_end|>
<|im_start|>assistant
每只猴子能吃到的桃子数=总桃子数/猴子的数量，84/6=14。所以答案是：14。 \boxed{14}<|im_end|>
<|im_start|>user
用面包机烤面包时，第一面烤2分钟，第二面只要烤1分钟，即烤一片面包需要3分钟，小勤的面包机一次只能放2片，他每天早上吃3片面包，至少需要烤多少分钟？<|im_end|>
<|im_start|>assistant
可以现将两片面包放入面包机烤2分钟，再将其中一片拿出来，将第三片面包放进去，烤1分钟，这样第一片面包就烤好了，将第一片面包拿出来将第二片面包放进去，继续烤1分钟，于是第二片面包也烤好了将其拿出来，第三片面包再烤1分钟也就烤好了，一共是2+1+1=5。所以答案是：5。 \boxed{5}<|im_end|>
<|im_start|>user
一组学生植树，每人栽6棵还剩4棵；如果其中3人各栽5棵，其余每人各栽7棵，正好栽完。这一组学生有多少人？<|im_end|>
<|im_start|>assistant
假设学生的数量是x，每人栽6棵还剩4棵，也就是说树苗的数量=6x+4，又知道如果其中3人各栽5棵，其余每人各栽7棵，正好栽完，即6x+4=3*5+(x-3)*7，化简方程得到：x=10。所以答案是：10。 \box

In [ ]:
import re
from datasets import load_dataset
from typing import List, Dict, Tuple
from evaluation.tasks.base_task import BaseEvalTask
from evaluation.metrics import check_correctness

class GaoKaoClozeTask(BaseEvalTask):
    def __init__(self, num_shots: int = 5):
        super().__init__(dataset_name="hails/agieval-gaokao-mathcloze", split="test", num_shots=num_shots)
        
        self.system_prompt = "你是一个高考数学专家。请解答这道填空题，并给出严谨的推导过程。最终的填空答案请放在 \\boxed{} 中。"
        
        self.golden_shots = [
            {
                "question": "设数列 $\\left\\{ a_{n} \\right\\}$ 的前 $n$ 项和为 $S_{n}$，且 $a_{1}=-1$，$a_{n+1}=S_{n+1} S_{n}$，则 $S_{n}=$ (\\quad).", 
                "answer": "让我们写出这个数列的前n项和：\n$S_n = a_1 + a_2 + ... + a_n$\n$S_n = -1 + (S_2 S_1) + (S_3 S_2) + ... + (S_{n+1} S_n)$\n$S_n = -1 + S_n (S_{n+1} - S_1) = -1 - S_n - S_n S_{n+1} = -1 - S_n$\n$S_n (1 - S_{n+1}) = -1 - S_n$\n$S_n = -\\frac{1}{1 - S_{n+1}}$\n因为这个数列后面的所有项都是0，我们可以看到对于所有$n \\geq 1$，\n$S_{n+1} = 0$。因此，我们有：\n$S_n = -\\frac{1}{1 - S_{n+1}} = -\\frac{1}{1 - 0} = -1$\n这个数列前n项和的公式是$S_n = -\\frac{1}{n}$。\n答案是 $-\\frac{1}{n}$ \\boxed{-\\frac{1}{n}}"
            },
            {
                "question": "若 $\\left(x+\\frac{1}{x}\\right)^{n}$ 的展开式中第 3 项与第 7 项的二项式系数相等，则该展 开式中 $\\frac{1}{x^{2}}$ 的系数为 (\\quad).", 
                "answer": "由题意可得，$C_{n}^{2} = C_{n}^{6}$\n\\therefore n=8\n展开式的通项 $T_{r+1} = C_{8}^{r} x^{8-r} \\left(\\frac{1}{x}\\right)^{r} = C_{8}^{r} x^{8-2r}$\n令 $8-2r = -2$ 可得 $r=5$\n此时系数为 $C_{8}^{5} = 56$\n答案是 56 \\boxed{56}"
            },
            {
                "question": "函数 $\\mathrm{f}(\\mathrm{x})=\\sin (\\mathrm{x}+2 \\phi)-2 \\sin \\phi \\cos (\\mathrm{x}+\\phi)$ 的最大值为 (\\quad).", 
                "answer": "函数 $f(x) = \\sin(x+2\\phi) - 2\\sin\\phi \\cos(x+\\phi) = \\sin[(x+\\phi)+\\phi] - 2\\sin\\phi \\cos(x+\\phi)$\n$= \\sin(x+\\phi)\\cos\\phi + \\cos(x+\\phi)\\sin\\phi - 2\\sin\\phi \\cos(x+\\phi) = \\sin(x+\\phi)\\cos\\phi - \\cos(x+\\phi)\\sin\\phi$\n$= \\sin[(x+\\phi)-\\phi] = \\sin x$\n故函数 $f(x)$ 的最大值为 1\n答案是 1 \\boxed{1}"
            },
            {
                "question": "已知向量 $\\vec{a}=(3,1)$，$\\vec{b}=(1,0)$，$\\vec{c}=\\vec{a}+k \\vec{b}$。若 $\\vec{a} \\perp \\vec{c}$，则 $k=$ (\\quad).", 
                "answer": "因为 $\\vec{a}=(3,1)$，$\\vec{b}=(1,0)$，所以 $\\vec{c} = \\vec{a} + k\\vec{b} = (3+k, 1)$。\n因为 $\\vec{a} \\perp \\vec{c}$，所以 $\\vec{a} \\cdot \\vec{c} = 3(3+k) + 1 \\times 1 = 0$，解得 $k=-\\frac{10}{3}$\n答案是 $-\\frac{10}{3}$ \\boxed{-\\frac{10}{3}}"
            },
            {
                "question": "设向量 $\\vec{a}$，$\\vec{b}$ 不平行，向量 $\\lambda \\vec{a}+\\vec{b}$ 与 $\\vec{a}+2 \\vec{b}$ 平行，则实数 $\\lambda=$ (\\quad).", 
                "answer": "因为向量 $\\vec{a}$，$\\vec{b}$ 不平行，向量 $\\lambda \\vec{a}+\\vec{b}$ 与 $\\vec{a}+2 \\vec{b}$ 平行，\n所以 $\\lambda \\vec{a}+\\vec{b} = t(\\vec{a}+2 \\vec{b}) = t \\vec{a} + 2t \\vec{b}$\n所以 $\\left\\{ \\begin{array}{l} \\lambda = t \\\\ 1 = 2t \\end{array} \\right.$\n解得实数 $\\lambda = \\frac{1}{2}$。\n答案是 $\\frac{1}{2}$ \\boxed{\\frac{1}{2}}"
            }
        ]

    def load_data(self) -> Tuple[List[Dict], List[str]]:
        try: 
            ds = load_dataset(self.dataset_name, split=self.split)
            return [{"question": item["query"]} for item in ds], [item["answer"] for item in ds]
        except Exception as e:
            print(f"Lỗi khi load {self.dataset_name}: {e}. Vui lòng kiểm tra lại dataset.")
            return [], []

    def build_prompt(self, problem: str, few_shot_examples: List[Dict]) -> str:
        prompt = f"<|im_start|>system\n{self.system_prompt}<|im_end|>\n"
        for ex in few_shot_examples:
            prompt += f"<|im_start|>user\n问题：\n{ex['question']}<|im_end|>\n<|im_start|>assistant\n解析：\n{ex['answer']}<|im_end|>\n"
        prompt += f"<|im_start|>user\n问题：\n{problem}<|im_end|>\n<|im_start|>assistant\n解析：\n"
        return prompt

    def evaluate_correctness(self, prediction: str, gold_answer: str) -> bool:
        # \boxed{}
        is_correct = check_correctness(prediction, gold_answer)
        if is_correct:
            return True
            
        # fallback
        fallback_pattern = r"答案是\s*([^\n\r]+)"
        matches = re.findall(fallback_pattern, prediction)
        
        if matches:
            pred_ans = matches[-1].strip()
            clean_pred = pred_ans.replace(" ", "").replace("$", "")
            clean_gold = gold_answer.replace(" ", "").replace("$", "")
            return clean_pred == clean_gold
            
        return False

<|im_start|>system
你是一个高考数学专家。请解答这道填空题，并给出严谨的推导过程。最终的填空答案请放在 \boxed{} 中。<|im_end|>
<|im_start|>user
问题：
设数列 $\left\{ a_{n} \right\}$ 的前 $n$ 项和为 $S_{n}$，且 $a_{1}=-1$，$a_{n+1}=S_{n+1} S_{n}$，则 $S_{n}=$ (\quad).<|im_end|>
<|im_start|>assistant
解析：
让我们写出这个数列的前n项和：
$S_n = a_1 + a_2 + ... + a_n$
$S_n = -1 + (S_2 S_1) + (S_3 S_2) + ... + (S_{n+1} S_n)$
$S_n = -1 + S_n (S_{n+1} - S_1) = -1 - S_n - S_n S_{n+1} = -1 - S_n$
$S_n (1 - S_{n+1}) = -1 - S_n$
$S_n = -\frac{1}{1 - S_{n+1}}$
因为这个数列后面的所有项都是0，我们可以看到对于所有$n \geq 1$，
$S_{n+1} = 0$。因此，我们有：
$S_n = -\frac{1}{1 - S_{n+1}} = -\frac{1}{1 - 0} = -1$
这个数列前n项和的公式是$S_n = -\frac{1}{n}$。
答案是 $-\frac{1}{n}$ \boxed{-\frac{1}{n}}<|im_end|>
<|im_start|>user
问题：
若 $\left(x+\frac{1}{x}\right)^{n}$ 的展开式中第 3 项与第 7 项的二项式系数相等，则该展 开式中 $\frac{1}{x^{2}}$ 的系数为 (\quad).<|im_end|>
<|im_start|>assistant
解析：
由题意可得，$C_{n}^{2} = C_{n}^{6}$
\therefore n=8
展开式的通项 $T_{r+1} = C_{8}^{r} x^{8-r} \left(\frac{1}{x}\right)^{r} = C_{8}^{r} x^{8-2r}$
令 $8-2r = -2$ 可得 $r=5$
此时系数为 $C_{

In [ ]:
import re
from datasets import load_dataset
from typing import List, Dict, Tuple
from evaluation.tasks.base_task import BaseEvalTask

class GaoKaoQATask(BaseEvalTask):
    def __init__(self, num_shots: int = 4):
        super().__init__(dataset_name="hails/agieval-gaokao-mathqa", split="test", num_shots=num_shots)

        self.system_prompt = "你是一个高考数学专家。仔细阅读下面的选择题，给出严谨的推导过程，并选出正确的选项 (A, B, C 或 D)."
        
        self.golden_shots = [
            {
                "question": "下列有关命题的说法正确的是( )\nA. 命题“若$ x^{2}=1 $, 则$ x=1 $”的否命题为：“若$ x^{2}=1 $, 则$ x\\neq 1 $” \nB. 命题“若$ x=y $, 则$ \\sin x=\\sin y $”的逆否命题为真命题 \nC. 命题“存在$ x\\in R $, 使得$ x^{2}+x+1 < 0 $”的否定是：“对任意$ x\\in R $, 均有$ x^{2}+x+1 < 0 $” \nD. “$ x=-1 $”是“$ x^{2}-5x-6=0 $”的必要不充分条件", 
                "answer": "命题“若$ x^{2}=1 $,则$ x=1 $”的否命题为“若$ x^{2}\\neq 1 $,则$ x\\neq 1 $”,故排除A;\n$\\because$命题“若$ x=y $,则$ \\sin x=\\sin y $”为真命题,故其逆否命题为真命题,B正确;\n命题“存在$ x\\in R $,使得$ x^{2}+x+1 < 0 $”的否定是:“对任意$ x\\in R $,均有$ x^{2}+x+1 \\geqslant 0 $”,故排除C;\n$\\because$“$ x^{2}-5x-6=0 $” $\\Leftrightarrow$ “$ x=-1 $或$ x=6 $”,$\\therefore$“$ x=-1 $”是“$ x^{2}-5x-6=0 $”的充分不必要条件,排除D;\n故选:B.\n推理结束。"
            },
            {
                "question": "已知函数$ f(x)=2x^{2}+mx-1 $,若对于任意$ x\\in[m,m+1] $,都有$ f(x)<0 $成立,则实数$ m $的取值范围是( )\nA. $ \\left(-\\sqrt{2},0\\right] $ \nB. $ \\left(-2,0\\right) $ \nC. $ \\left[-\\dfrac{\\sqrt{2}}{2},0\\right] $ \nD. $ \\left(-\\dfrac{\\sqrt{2}}{2},0\\right) $", 
                "answer": "由题意可得$\\begin{cases}f(m)=2m^{2}-1 < 0 \\\\ f(m+1)=2(m+1)^{2}+m(m+1)-1 < 0\\end{cases}$, \n求得$ -\\dfrac{\\sqrt{2}}{2} < m < 0 $,\n即实数$ m $的取值范围为$ \\left(-\\dfrac{\\sqrt{2}}{2},0\\right) $.\n故选:D.\n推理结束。"
            },
            {
                "question": "设$ i $是虚数单位,若复数$ a+\\dfrac{5i}{1-2i}(a\\in R) $是纯虚数,则$ a $等于( )\nA. -1 \nB. 1 \nC. 2 \nD. -2", 
                "answer": "$\\because a+\\dfrac{5i}{1-2i}=a+\\dfrac{5i(1+2i)}{(1-2i)(1+2i)}=a+\\dfrac{-10+5i}{5}=a-2+i$是纯虚数,\n$\\therefore a=2$.\n故选:C.\n推理结束。"
            },
            {
                "question": "已知集合$ A=\\{x|2\\leqslant x < 7\\} $, $ B=\\{x|3 < x < 10\\} $, $ C=\\{x|a-5 < x < a\\} $. 若非空集合$ C\\subseteq(A\\cup B) $,则$ a $的取值范围是( )\nA. $ 7\\leqslant a\\leqslant 10 $ \nB. $ 7\\leqslant a < 10 $ \nC. $ 8 < a < 10 $ \nD. $ 8\\leqslant a\\leqslant 10 $", 
                "answer": "$\\because$集合$ A=\\{x|2\\leqslant x < 7\\} $, $ B=\\{x|3 < x < 10\\} $,\n$\\therefore A\\cap B=\\{x|3 < x < 7\\} $,\n$ A\\cup B=\\{x|2\\leqslant x < 10\\} $,\n当$ C\\neq \\varnothing $时,要使$ C\\subseteq(A\\cup B) $,$\\begin{cases}a-5\\geqslant 2 \\\\ a\\leqslant 10\\end{cases}$,解得$ 7\\leqslant a\\leqslant 10 $;\n$\\therefore a $的取值范围是$ 7\\leqslant a\\leqslant 10 $.\n故选:A.\n推理结束。"
            }
        ]

    def load_data(self) -> Tuple[List[Dict], List[str]]:
        try: 
            ds = load_dataset(self.dataset_name, split=self.split)
            examples = []
            gold_answers = []
            
            idx_to_char = {0: "A", 1: "B", 2: "C", 3: "D"}
            
            for item in ds:
                q_text = item["query"]
                
                # bỏ prefix "问题：" ở đầu (nếu có)
                if q_text.startswith("问题：") or q_text.startswith("问题:"):
                    q_text = q_text[3:].strip()
                    
                examples.append({"question": q_text})
                
                gold_idx = int(item["gold"][0]) 
                gold_answers.append(idx_to_char.get(gold_idx, "A")) # fallback
                
            return examples, gold_answers
            
        except Exception as e:
            print(f"Lỗi khi load {self.dataset_name}: {e}. Vui lòng kiểm tra lại dataset.")
            return [], []

    def build_prompt(self, problem: str, few_shot_examples: List[Dict]) -> str:
        prompt = f"<|im_start|>system\n{self.system_prompt}<|im_end|>\n"
        for ex in few_shot_examples:
            prompt += f"<|im_start|>user\n选择题: {ex['question']}<|im_end|>\n<|im_start|>assistant\n解:{ex['answer']}<|im_end|>\n"
        prompt += f"<|im_start|>user\n选择题: {problem}<|im_end|>\n<|im_start|>assistant\n解:"
        return prompt

    def evaluate_correctness(self, prediction: str, gold_answer: str) -> bool:
        specific_pattern = r"故选\s*[:：]?\s*([A-D])"
        matches = re.findall(specific_pattern, prediction.upper())
        if matches:
            return matches[-1] == gold_answer
            
        fallback_pattern = r'\b([A-D])\b'
        fallback_matches = re.findall(fallback_pattern, prediction.upper())
        if fallback_matches:
            return fallback_matches[-1] == gold_answer
            
        return False

<|im_start|>system
你是一个高考数学专家。仔细阅读下面的选择题，给出严谨的推导过程，并选出正确的选项 (A, B, C 或 D)。<|im_end|>
<|im_start|>user
选择题: 下列有关命题的说法正确的是( )
A. 命题“若$ x^{2}=1 $, 则$ x=1 $”的否命题为：“若$ x^{2}=1 $, 则$ x\neq 1 $” 
B. 命题“若$ x=y $, 则$ \sin x=\sin y $”的逆否命题为真命题 
C. 命题“存在$ x\in R $, 使得$ x^{2}+x+1 < 0 $”的否定是：“对任意$ x\in R $, 均有$ x^{2}+x+1 < 0 $” 
D. “$ x=-1 $”是“$ x^{2}-5x-6=0 $”的必要不充分条件<|im_end|>
<|im_start|>assistant
解:命题“若$ x^{2}=1 $,则$ x=1 $”的否命题为“若$ x^{2}\neq 1 $,则$ x\neq 1 $”,故排除A;
$\because$命题“若$ x=y $,则$ \sin x=\sin y $”为真命题,故其逆否命题为真命题,B正确;
命题“存在$ x\in R $,使得$ x^{2}+x+1 < 0 $”的否定是:“对任意$ x\in R $,均有$ x^{2}+x+1 \geqslant 0 $”,故排除C;
$\because$“$ x^{2}-5x-6=0 $” $\Leftrightarrow$ “$ x=-1 $或$ x=6 $”,$\therefore$“$ x=-1 $”是“$ x^{2}-5x-6=0 $”的充分不必要条件,排除D;
故选:B.
推理结束。<|im_end|>
<|im_start|>user
选择题: 已知函数$ f(x)=2x^{2}+mx-1 $,若对于任意$ x\in[m,m+1] $,都有$ f(x)<0 $成立,则实数$ m $的取值范围是( )
A. $ \left(-\sqrt{2},0\right] $ 
B. $ \left(-2,0\right) $ 
C. $ \left[-\dfrac{\sqrt{2}}{2},0\right] $ 
D. $ \left(-\dfrac{\sqrt{2}}{2},0\right

In [ ]:
import re
from datasets import load_dataset
from typing import List, Dict, Tuple
from evaluation.tasks.base_task import BaseEvalTask

class GSM8KTask(BaseEvalTask):
    def __init__(self, num_shots: int = 8):
        super().__init__(dataset_name="openai/gsm8k", split="test", num_shots=num_shots)
        self.system_prompt = "You are a helpful math assistant."
        
        self.golden_shots = [
            {
                "question": "In 2004, there were 60 kids at a cookout. In 2005, half the number of kids came to the cookout as compared to 2004. In 2006, 2/3 as many kids came to the cookout as in 2005. How many kids came to the cookout in 2006?", 
                "answer": "In 2005, 60/2=30 kids came to the cookout.\nIn 2006, 30/3*2=20 kids came to the cookout.\nThe answer is 20"
            },
            {
                "question": "Zilla spent 7% of her monthly earnings on rent, half of it on her other monthly expenses, and put the rest in her savings. If she spent $133 on her rent, how much does she deposit into her savings account in a month?", 
                "answer": "Since $133 is equal to 7% of her earnings, then 1% is equal to $133/7 = $19.\nThe total monthly earning of Zilla is represented by 100%, so $19 x 100 = $1900 is her monthly earnings.\nSo, $1900/2 = $950 is spent on her other monthly expenses.\nThe total amount spent on the rent and other monthly expenses is $133 + $950 = $1083.\nHence, she saves $1900 - $1083 = $817 per month.\nThe answer is 817"
            },
            {
                "question": "If Buzz bought a pizza with 78 slices at a restaurant and then decided to share it with the waiter in the ratio of 5:8, with Buzz's ratio being 5, what's twenty less the number of slices of pizza that the waiter ate?", 
                "answer": "The total ratio representing the slices of pizza that Buzz bought is 5+8=13\nIf he shared the slices of pizza with the waiter, the waiter received a fraction of 8/13 of the total number of slices, which totals 8/13 * 78 = 48 slices\nTwenty less the number of slices of pizza that the waiter ate is 48-20 = 28\nThe answer is 28"
            },
            {
                "question": "Jame gets a raise to $20 per hour and works 40 hours a week. His old job was $16 an hour for 25 hours per week. How much more money does he make per year in his new job than the old job if he works 52 weeks a year?", 
                "answer": "He makes 20*40=$800 per week\nHe used to make 16*25=$400 per week\nSo his raise was 800-400=$400 per week\nSo he makes 400*52=$20,800 per year more\nThe answer is 20800"
            },
            {
                "question": "Mr. Gardner bakes 20 cookies, 25 cupcakes, and 35 brownies for his second-grade class of 20 students. If he wants to give each student an equal amount of sweet treats, how many sweet treats will each student receive?", 
                "answer": "Mr. Gardner bakes a total of 20 + 25 + 35 = 80 sweet treats\nEach student will receive 80 / 20 = 4 sweet treats\nThe answer is 4"
            },
            {
                "question": "A used car lot has 24 cars and motorcycles (in total) for sale. A third of the vehicles are motorcycles, and a quarter of the cars have a spare tire included. How many tires are on the used car lot's vehicles in all?", 
                "answer": "The used car lot has 24 / 3 = 8 motorcycles with 2 tires each.\nThe lot has 24 - 8 = 16 cars for sale\nThere are 16 / 4 = 4 cars with a spare tire with 5 tires each.\nThe lot has 16 - 4 = 12 cars with 4 tires each.\nThus, the used car lot's vehicles have 8 * 2 + 4 * 5 + 12 * 4 = 16 + 20 + 48 = 84 tires in all.\nThe answer is 84"
            },
            {
                "question": "Norma takes her clothes to the laundry. She leaves 9 T-shirts and twice as many sweaters as T-shirts in the washer. When she returns she finds 3 sweaters and triple the number of T-shirts. How many items are missing?", 
                "answer": "Norma left 9 T-shirts And twice as many sweaters, she took 9 * 2= 18 sweaters\nAdding the T-shirts and sweaters, Norma left 9 + 18 = 27 clothes\nWhen she came back, she found 3 sweaters And triple the number of T-shirts, she found 3 * 3 = 9 T-shirts\nAdding the T-shirts and sweaters, Norma found 3 + 9 = 12 clothes\nSubtracting the clothes she left from the clothes she found, 27 - 12 = 15 clothes are missing\nThe answer is 15"
            },
            {
                "question": "Adam has an orchard. Every day for 30 days he picks 4 apples from his orchard. After a month, Adam has collected all the remaining apples, which were 230. How many apples in total has Adam collected from his orchard?", 
                "answer": "During 30 days Adam picked 4 * 30 = 120 apples.\nSo in total with all the remaining apples, he picked 120 + 230 = 350 apples from his orchard.\nThe answer is 350"
            }
        ]

    def load_data(self) -> Tuple[List[Dict], List[str]]:
        ds = load_dataset(self.dataset_name, "main", split=self.split)
        examples = [{"question": item["question"]} for item in ds]
        # GSM8K lưu đáp án sau chuỗi "#### "
        gold_answers = [item["answer"].split("####")[-1].strip().replace(",", "") for item in ds]
        return examples, gold_answers

    def build_prompt(self, problem: str, few_shot_examples: List[Dict]) -> str:
        prompt = f"<|im_start|>system\n{self.system_prompt}<|im_end|>\n"
        
        for ex in few_shot_examples:
            prompt += f"<|im_start|>user\nQuestion: {ex['question']}<|im_end|>\n"
            prompt += f"<|im_start|>assistant\nLet's think step by step\n{ex['answer']}<|im_end|>\n"

        # question    
        prompt += f"<|im_start|>user\nQuestion: {problem}<|im_end|>\n"
        prompt += f"<|im_start|>assistant\nLet's think step by step\n"
        return prompt

    def evaluate_correctness(self, prediction: str, gold_answer: str) -> bool:

        matches = re.findall(r"The answer is\s*([0-9,\.\-]+)", prediction, re.IGNORECASE)
        if matches:
            pred_ans = matches[-1].replace(",", "").strip()
            return pred_ans == gold_answer
            
        # fallback
        nums = re.findall(r'-?\d+\.?\d*', prediction.replace(",", ""))
        if nums:
            pred_ans = nums[-1]
            return pred_ans == gold_answer
            
        return False

<|im_start|>system
You are a helpful math assistant.<|im_end|>
<|im_start|>user
Question: In 2004, there were 60 kids at a cookout. In 2005, half the number of kids came to the cookout as compared to 2004. In 2006, 2/3 as many kids came to the cookout as in 2005. How many kids came to the cookout in 2006?<|im_end|>
<|im_start|>assistant
Let's think step by step
In 2005, 60/2=30 kids came to the cookout.
In 2006, 30/3*2=20 kids came to the cookout.
The answer is 20<|im_end|>
<|im_start|>user
Question: Zilla spent 7% of her monthly earnings on rent, half of it on her other monthly expenses, and put the rest in her savings. If she spent $133 on her rent, how much does she deposit into her savings account in a month?<|im_end|>
<|im_start|>assistant
Let's think step by step
Since $133 is equal to 7% of her earnings, then 1% is equal to $133/7 = $19.
The total monthly earning of Zilla is represented by 100%, so $19 x 100 = $1900 is her monthly earnings.
So, $1900/2 = $950 is spent on her oth

In [ ]:
import re
from datasets import load_dataset, concatenate_datasets
from typing import List, Dict, Tuple
from evaluation.tasks.base_task import BaseEvalTask
from evaluation.metrics import check_correctness

class MathTask(BaseEvalTask):
    def __init__(self, num_shots: int = 4):
        super().__init__(dataset_name="EleutherAI/hendrycks_math", split="test", num_shots=num_shots)

        self.system_prompt = "You are a mathematics expert. Solve the problem step by step."
        
        # 7 subset
        self.subsets = [
            "algebra",
            "counting_and_probability",
            "geometry",
            "intermediate_algebra",
            "number_theory",
            "prealgebra",
            "precalculus"
        ]
        
        self.golden_shots = [
            {
                "question": "Find the domain of the expression \\frac{\\sqrt{x-2}}{\\sqrt{5-x}}.", 
                "answer": "The expressions inside each square root must be non-negative. Therefore, $x-2 \\ge 0$, so $x\\ge2$, and $5 - x \\ge 0$, so $x \\le 5$. Also, the denominator cannot be equal to zero, so $5-x>0$, which gives $x<5$. Therefore, the domain of the expression is $\\boxed{[2,5)}$.\nThe answer is: $[2,5)$."
            },
            {
                "question": "If $\\det \\mathbf{A} = 2$ and $\\det \\mathbf{B} = 12,$ then find $\\det (\\mathbf{A} \\mathbf{B}).$", 
                "answer": "We have that $\\det (\\mathbf{A} \\mathbf{B}) = (\\det \\mathbf{A})(\\det \\mathbf{B}) = (2)(12) = \\boxed{24}.$\nThe answer is: $24$."
            },
            {
                "question": "Terrell usually lifts two 20-pound weights 12 times. If he uses two 15-pound weights instead, how many times must Terrell lift them in order to lift the same total weight?", 
                "answer": "If Terrell lifts two 20-pound weights 12 times, he lifts a total of $2\\cdot 12\\cdot 20=480$ pounds of weight. If he lifts two 15-pound weights instead for $n$ times, he will lift a total of $2\\cdot 15\\cdot n=30n$ pounds of weight.\nEquating this to 480 pounds, we can solve for $n$:\n\\begin{align*}\n30n&=480\\\\\n\\Rightarrow\\qquad n&=480/30=\\boxed{16}\n\\end{align*}\nThe answer is: $16$."
            },
            {
                "question": "If the system of equations\n\\begin{align*}\n6x-4y&=a,\\\\\n6y-9x&=b.\n\\end{align*} has a solution $(x, y)$ where $x$ and $y$ are both nonzero, find $\\frac{a}{b},$ assuming $b$ is nonzero.", 
                "answer": "If we multiply the first equation by $-\\frac{3}{2}$, we obtain $$6y-9x=-\\frac{3}{2}a.$$ Since we also know that $6y-9x=b$, we have $$-\\frac{3}{2}a=b\\Rightarrow\\frac{a}{b}=\\boxed{-\\frac{2}{3}}.$$\nThe answer is: $-\\frac{2}{3}$."
            }
        ]

    def load_data(self) -> Tuple[List[Dict], List[str]]:
        examples = []
        gold_answers = []
        
        for subset in self.subsets:
            try:
                ds = load_dataset(self.dataset_name, subset, split=self.split)
                
                for item in ds:
                    examples.append({"question": item["problem"]})
                    gold_answers.append(item["solution"])
                    
            except Exception as e:
                print(f"Lỗi khi load subset {subset}: {e}")
                
        return examples, gold_answers

    def build_prompt(self, problem: str, few_shot_examples: List[Dict]) -> str:
        prompt = f"<|im_start|>system\n{self.system_prompt}<|im_end|>\n"
        
        for ex in few_shot_examples:
            prompt += f"<|im_start|>user\nProblem:\n{ex['question']}<|im_end|>\n"
            prompt += f"<|im_start|>assistant\nSolution:\n{ex['answer']}<|im_end|>\n"
            
        # question
        prompt += f"<|im_start|>user\nProblem:\n{problem}\n<|im_end|>\n"
        prompt += f"<|im_start|>assistant\nSolution:\n"
        
        return prompt

    def evaluate_correctness(self, prediction: str, gold_answer: str) -> bool:

        # \boxed{}
        is_correct = check_correctness(prediction, gold_answer)
        if is_correct:
            return True
            
        # fallback
        import re
        fallback_pattern = r"The answer is:\s*\$?([^\$\n]+)\$?"
        matches = re.findall(fallback_pattern, prediction)
        
        if matches:
            pred_ans = matches[-1].strip()
            if pred_ans.endswith("."):
                pred_ans = pred_ans[:-1]
                
            from evaluation.metrics import normalize_math_string
            return normalize_math_string(pred_ans) == normalize_math_string(gold_answer)
            
        return False

<|im_start|>system
You are a mathematics expert. Solve the problem step by step.<|im_end|>
<|im_start|>user
Problem:
Find the domain of the expression \frac{\sqrt{x-2}}{\sqrt{5-x}}.<|im_end|>
<|im_start|>assistant
Solution:
The expressions inside each square root must be non-negative. Therefore, $x-2 \ge 0$, so $x\ge2$, and $5 - x \ge 0$, so $x \le 5$. Also, the denominator cannot be equal to zero, so $5-x>0$, which gives $x<5$. Therefore, the domain of the expression is $\boxed{[2,5)}$.
The answer is: $[2,5)$.<|im_end|>
<|im_start|>user
Problem:
If $\det \mathbf{A} = 2$ and $\det \mathbf{B} = 12,$ then find $\det (\mathbf{A} \mathbf{B}).$<|im_end|>
<|im_start|>assistant
Solution:
We have that $\det (\mathbf{A} \mathbf{B}) = (\det \mathbf{A})(\det \mathbf{B}) = (2)(12) = \boxed{24}.$
The answer is: $24$.<|im_end|>
<|im_start|>user
Problem:
Terrell usually lifts two 20-pound weights 12 times. If he uses two 15-pound weights instead, how many times must Terrell lift them in order to lift

In [ ]:
import re
from datasets import load_dataset
from typing import List, Dict, Tuple
from evaluation.tasks.base_task import BaseEvalTask

class MMLUStemTask(BaseEvalTask):
    def __init__(self, num_shots: int = 4):
        super().__init__(dataset_name="cais/mmlu", split="test", num_shots=num_shots)
        self.system_prompt = "You are an expert in STEM subjects. Read the multiple-choice question, explain your reasoning, and choose the correct answer."
        self.choices = ["A", "B", "C", "D"]
        
        # 18 subset STEM
        self.stem_subjects = [
            "abstract_algebra", "astronomy", "college_biology", "college_chemistry",
            "college_computer_science", "college_mathematics", "college_physics",
            "computer_security", "conceptual_physics", "electrical_engineering",
            "elementary_mathematics", "high_school_biology", "high_school_chemistry",
            "high_school_computer_science", "high_school_mathematics", "high_school_physics",
            "high_school_statistics", "machine_learning"
        ]
        
        self.golden_shots = [
            {
                "question": "Find the domain of the expression \\frac{\\sqrt{x-2}}{\\sqrt{5-x}}.\nWhat of the following is the right choice? Explain your answer.\n(A) [-5,-2) \n(B) [2,5) \n(C) [-2,-5) \n(D) [5,2)", 
                "answer": "The expressions inside each square root must be non-negative. Therefore, $x-2 \\ge 0$, so $x\\ge2$, and $5 - x \\ge 0$, so $x \\le 5$. Also, the denominator cannot be equal to zero, so $5-x>0$, which gives $x<5$.\nTherefore, the domain of the expression is $\\boxed{[2,5)}$.\nFinal Answer: The final answer is (B). I hope it is correct."
            },
            {
                "question": "If $\\det \\mathbf{A} = 2$ and $\\det \\mathbf{B} = 12,$ then find $\\det (\\mathbf{A} \\mathbf{B}).$\nWhat of the following is the right choice? Explain your answer.\n(A) 14 \n(B) 4 \n(C) 2 \n(D) 24", 
                "answer": "We have that $\\det (\\mathbf{A} \\mathbf{B}) = (\\det \\mathbf{A})(\\det \\mathbf{B}) = (2)(12) = \\boxed{24}.$\nFinal Answer: The final answer is (D). I hope it is correct."
            },
            {
                "question": "Terrell usually lifts two 20-pound weights 12 times. If he uses two 15-pound weights instead, how many times must Terrell lift them in order to lift the same total weight?\nWhat of the following is the right choice? Explain your answer.\n(A) 12 \n(B) 20 \n(C) 16 \n(D) 15", 
                "answer": "If Terrell lifts two 20-pound weights 12 times, he lifts a total of $2\\cdot 12\\cdot 20=480$ pounds of weight. If he lifts two 15-pound weights instead for $n$ times, he will lift a total of $2\\cdot 15\\cdot n=30n$ pounds of weight.\nEquating this to 480 pounds, we can solve for $n$:\n\\begin{align*}\n30n&=480\\\\\n\\Rightarrow\\qquad n&=480/30=\\boxed{16}\n\\end{align*}\nFinal Answer: The final answer is (C). I hope it is correct."
            },
            {
                "question": "If the system of equations\n\\begin{align*}\n6x-4y&=a,\\\\\n6y-9x &=b.\n\\end{align*}\nhas a solution $(x, y)$ where $x$ and $y$ are both nonzero, find $\\frac{a}{b},$ assuming $b$ is nonzero.\nWhat of the following is the right choice? Explain your answer.\n(A) $-\\frac{2}{3}$ \n(B) $\\frac{2}{3}$ \n(C) $\\frac{1}{3}$ \n(D) $\\frac{4}{9}$", 
                "answer": "If we multiply the first equation by $-\\frac{3}{2}$, we obtain $$6y-9x=-\\frac{3}{2}a.$$ Since we also know that $6y-9x=b$, we have $$-\\frac{3}{2}a=b\\Rightarrow\\frac{a}{b}=\\boxed{-\\frac{2}{3}}.$$\nFinal Answer: The final answer is (A). I hope it is correct."
            }
        ]

    def load_data(self) -> Tuple[List[Dict], List[str]]:
        examples = []
        gold_answers = []
        
        for subset in self.stem_subjects:
            try:
                ds = load_dataset(self.dataset_name, subset, split=self.split)
                
                for item in ds:
                    q = item["question"] + "\nWhat of the following is the right choice? Explain your answer.\n"
                    for i, choice in enumerate(item["choices"]):
                        q += f"({self.choices[i]}) {choice}\n"
                    q = q.strip()
                        
                    examples.append({"question": q})
                    
                    gold_answers.append(self.choices[item["answer"]])
            except Exception as e:
                print(f"Lỗi khi load subset {subset}: {e}")
                
        return examples, gold_answers

    def build_prompt(self, problem: str, few_shot_examples: List[Dict]) -> str:
        prompt = f"<|im_start|>system\n{self.system_prompt}<|im_end|>\n"
        
        for ex in few_shot_examples:
            prompt += f"<|im_start|>user\nProblem:\n{ex['question']}<|im_end|>\n"
            prompt += f"<|im_start|>assistant\nSolution:\n{ex['answer']}<|im_end|>\n"
            
        prompt += f"<|im_start|>user\nProblem:\n{problem}\n<|im_end|>\n"
        prompt += f"<|im_start|>assistant\nSolution:\n"
        
        return prompt

    def evaluate_correctness(self, prediction: str, gold_answer: str) -> bool:

        specific_pattern = r"The final answer is \(([A-D])\)"
        matches = re.findall(specific_pattern, prediction)
        if matches:
            return matches[-1] == gold_answer
            
        # fallback 
        bracket_pattern = r"\(([A-D])\)"
        bracket_matches = re.findall(bracket_pattern, prediction)
        if bracket_matches:
            return bracket_matches[-1] == gold_answer
        
        fallback_pattern = r'\b([A-D])\b'
        fallback_matches = re.findall(fallback_pattern, prediction.upper())
        if fallback_matches:
            return fallback_matches[-1] == gold_answer
            
        return False

<|im_start|>system
You are an expert in STEM subjects. Read the multiple-choice question, explain your reasoning, and choose the correct answer.<|im_end|>
<|im_start|>user
Problem:
Find the domain of the expression \frac{\sqrt{x-2}}{\sqrt{5-x}}.
What of the following is the right choice? Explain your answer.
(A) [-5,-2) 
(B) [2,5) 
(C) [-2,-5) 
(D) [5,2)<|im_end|>
<|im_start|>assistant
Solution:
The expressions inside each square root must be non-negative. Therefore, $x-2 \ge 0$, so $x\ge2$, and $5 - x \ge 0$, so $x \le 5$. Also, the denominator cannot be equal to zero, so $5-x>0$, which gives $x<5$.
Therefore, the domain of the expression is $\boxed{[2,5)}$.
Final Answer: The final answer is (B). I hope it is correct.<|im_end|>
<|im_start|>user
Problem:
If $\det \mathbf{A} = 2$ and $\det \mathbf{B} = 12,$ then find $\det (\mathbf{A} \mathbf{B}).$
What of the following is the right choice? Explain your answer.
(A) 14 
(B) 4 
(C) 2 
(D) 24<|im_end|>
<|im_start|>assistant
Solution:
We ha

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import logging
from tqdm import tqdm

class MathEvaluator:
    def __init__(self, model_path: str, tensor_parallel_size: int = 1):
        """Khởi tạo HuggingFace Transformers engine."""
        self.log = logging.getLogger(__name__)
        self.log.info(f"Initializing HuggingFace Transformers from {model_path}...")
        
        if torch.cuda.is_available():
            self.device = "cuda"
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        elif torch.backends.mps.is_available():
            self.device = "mps"
            dtype = torch.float16
        else:
            self.device = "cpu"
            dtype = torch.float32

        self.log.info(f"Using device: {self.device}, dtype: {dtype}")

        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        if self.tokenizer.pad_token_id is None:
             self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map=self.device,
            torch_dtype=dtype,
            trust_remote_code=True
        )
        self.model.eval()

    def generate_answers(self, prompts: list[str], temperature: float = 0.0) -> list[str]:
        """Sinh câu trả lời cho danh sách prompts."""
        predictions = []
        
        print("Đang sinh câu trả lời...")
        for prompt in tqdm(prompts, desc="Generating"):
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
            
            gen_kwargs = {
                "max_new_tokens": 2048,
                "eos_token_id": self.tokenizer.eos_token_id,
                "pad_token_id": self.tokenizer.pad_token_id,
                "do_sample": temperature > 0.0,
            }
            if temperature > 0.0:
                 gen_kwargs["temperature"] = temperature
                 gen_kwargs["top_p"] = 0.9

            with torch.no_grad():
                outputs = self.model.generate(**inputs, **gen_kwargs)
            
            input_length = inputs.input_ids.shape[1]
            generated_ids = outputs[0][input_length:]
            
            response = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
            
            response = response.replace("<|im_end|>", "").strip()
            predictions.append(response)
            
        return predictions


In [ ]:
from tqdm.notebook import tqdm
# from evaluation.evaluator import MathEvaluator

# from evaluation.tasks.math_task import MathTask
# from evaluation.tasks.gsm8k_task import GSM8KTask
# from evaluation.tasks.mmlu_stem_task import MMLUStemTask
# from evaluation.tasks.cmath_task import CMATHTask
# from evaluation.tasks.gaokao_cloze_task import GaoKaoClozeTask
# from evaluation.tasks.gaokao_qa_task import GaoKaoQATask

TASKS = {
    "math": MathTask,
    "gsm8k": GSM8KTask,
    "mmlu_stem": MMLUStemTask,
    "cmath": CMATHTask,
    "gaokao_cloze": GaoKaoClozeTask,
    "gaokao_qa": GaoKaoQATask
}


class Config:
    model_path = "Qwen/Qwen2.5-Math-1.5B-Instruct" 
    task = "math"  # "gsm8k", "mmlu_stem"...
    tp = 1         # Tensor Parallel size (Nếu Colab có 1 GPU thì để 1)
    temperature = 0.0

def main():
    args = Config()
    
    TaskClass = TASKS[args.task]
    task = TaskClass()
    
    print(f"Bắt đầu đánh giá Task: {args.task.upper()} ({task.num_shots}-shot)")
    
    print("Đang tải dữ liệu...")
    raw_examples, gold_answers = task.load_data()
    
    if not raw_examples:
        print("Không có dữ liệu để đánh giá. Vui lòng kiểm tra lại dataset.")
        return

    # Golden Few-shots
    few_shot_examples = task.generate_few_shots()
    print(f"Đã nạp {len(few_shot_examples)} ví dụ Golden Few-shots.")

    prompts = [task.build_prompt(ex["question"], few_shot_examples) for ex in raw_examples]

    evaluator = MathEvaluator(model_path=args.model_path, tensor_parallel_size=args.tp)
    print("Đang sinh câu trả lời...")
    predictions = evaluator.generate_answers(prompts, temperature=args.temperature)

    print("Evaluating...")
    correct = 0
    for pred, gold in tqdm(zip(predictions, gold_answers), total=len(predictions)):
        if task.evaluate_correctness(pred, gold):
            correct += 1

    accuracy = (correct / len(prompts)) * 100
    print("\n" + "="*50)
    print("BENCHMARK")
    print("="*50)
    print(f"Model    : {args.model_path}")
    print(f"Task     : {args.task.upper()} ({task.num_shots}-shot)")
    print(f"Accuracy : {accuracy:.2f}% ({correct}/{len(prompts)})")
    print("="*50)


main()

Bắt đầu đánh giá Task: MATH (4-shot)
Đang tải dữ liệu...
Bắt đầu tải dữ liệu từ 7 subsets của MATH...
 - Đang tải subset: algebra (split=test)
 - Đang tải subset: counting_and_probability (split=test)
 - Đang tải subset: geometry (split=test)
 - Đang tải subset: intermediate_algebra (split=test)
 - Đang tải subset: number_theory (split=test)
 - Đang tải subset: prealgebra (split=test)
 - Đang tải subset: precalculus (split=test)
✅ Đã tải xong tổng cộng 5000 câu hỏi từ MATH.
Đã nạp 4 ví dụ Golden Few-shots.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Đang sinh câu trả lời bằng vLLM...
Đang sinh câu trả lời...


Generating:   0%|          | 0/5000 [00:00<?, ?it/s]